# Phase 1: Exploratory Data Analysis (EDA) - FIFA Player Value Dataset

This notebook conducts a systematic audit of the seven FIFA Career Mode editions (FIFA 15 through FIFA 21) across schema integrity, missingness patterns, zero-value anomalies, scale drift, attrition, and history length distributions.

In [1]:
import os, tempfile
os.environ['MPLCONFIGDIR'] = tempfile.gettempdir()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from fifa_value.data.load import load_all_seasons
from fifa_value.data.clean import clean_panel_data

In [2]:
df_raw = load_all_seasons(raw_dir='../data/raw')
print(f'Total records loaded: {len(df_raw):,}')
print(f'Number of seasons: {df_raw["season"].nunique()}')
df_raw.groupby('season').size()

## 1. Schema & Missingness Audit
- Outfield skills (`pace`...`physic`) are absent for goalkeepers (~11% overall).
- Goalkeeper skills (`gk_diving`...`gk_speed`) are absent for outfielders (~89% overall).
- `defending_marking` is 100% missing in FIFA 21 (dropped).
- `release_clause_eur` is 100% missing in FIFA 15 (modeled with indicator).

In [3]:
p21 = df_raw[df_raw['season'] == 2021]
p15 = df_raw[df_raw['season'] == 2015]
print('FIFA 21 defending_marking missing rate:', p21['defending_marking'].isna().mean())
print('FIFA 15 release_clause_eur missing rate:', p15['release_clause_eur'].isna().mean())

## 2. Zero-Value Player Profiling
Confirming the club-less hypothesis: >95% of €0 value players in FIFA 21 have no club and zero wage.

In [4]:
p21_zero = p21[p21['value_eur'] == 0]
print('Total zero value players in FIFA 21:', len(p21_zero))
print('Percentage clubless:', p21_zero['club_name'].isna().mean() * 100)
print('Percentage zero wage:', (p21_zero['wage_eur'] == 0).mean() * 100)
p21_zero[['short_name', 'overall', 'club_name', 'wage_eur']].head(10)

## 3. Scale Drift & Attrition Analysis

In [5]:
drift = df_raw[df_raw['value_eur'] > 0].groupby('season')['value_eur'].agg(['median', lambda x: x.quantile(0.90), 'mean'])
drift.columns = ['Median Value', '90th Percentile', 'Mean Value']
drift / 1e6